# Demo 03: Distillation Pipeline — Teacher → Student

**Block 5** | Day 1 | 25 minutes

This notebook demonstrates the **knowledge distillation** pipeline: a large foundation
model (the teacher) auto-labels images, and a small specialized model (the student)
trains on those labels. The teacher retires. The student deploys.

We use **SAM3** (Segment Anything Model 3, via HuggingFace Transformers) as the teacher
and **YOLO26n** (2.4M parameters) as the student. Most training results are pre-baked
because live training takes 30+ minutes.

---
## The Distillation Concept

```
TEACHER (SAM3 / YOLOE)                    STUDENT (YOLO26n)
• 2.9B / 44.9M parameters                 • 2.4M parameters
• Slow, accurate, open vocabulary         • Fast, specialized, fixed classes
• Knows everything                        • Knows YOUR task
• Too big for edge                        • Runs on Jetson / mobile
         ──── auto-labels ────→
              (YOLO .txt files)
```

The bridge between them is **auto-labeling**. The teacher looks at your images, draws
bounding boxes, writes YOLO-format label files. The student trains on those labels.
The teacher retires.

The student doesn't know it was taught by a machine. The label format is identical to
human annotations. Same `.txt` files, same bounding box coordinates.

---
## Why Not Deploy the Teacher?

| Metric | Teacher (YOLOE-L) | Student (YOLO26n) | Difference |
|--------|:-----------------:|:-----------------:|:----------:|
| **Parameters** | 44.9M | 2.4M | 19x smaller |
| **Inference (T4)** | 6.2 ms | 1.7 ms | 3.6x faster |
| **Inference (Jetson)** | ~40 ms (25 FPS) | ~11 ms (90 FPS) | Below vs above 30 FPS threshold |
| **Weights size** | 79 MB | 5.5 MB | Fits on microcontroller |
| **Cost at scale** | ~$0.001/image | ~$0.0001/image | 10x cheaper |
| **Vocabulary** | Open (any text prompt) | Fixed (trained classes only) | Flexibility vs speed |

**The teacher's job is to teach, not to deploy.**

On a Jetson, YOLOE-L drops below the 30 FPS threshold for real-time camera systems.
YOLO26n runs at 90 FPS -- three times the requirement. Same detections. Same classes.
Fraction of the compute.

---
## Step 1: Load the Teacher Model (SAM3)

In [ ]:
import torch
import sys
import time
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np

# Device detection: CUDA > MPS > CPU
if torch.cuda.is_available():
    device = "cuda"
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print(f"Python:  {sys.version}")
print(f"PyTorch: {torch.__version__}")
print(f"Device:  {device}")
if device == "cuda":
    print(f"GPU:     {torch.cuda.get_device_name(0)}")
    print(f"VRAM:    {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# ---- Data paths ----
# Works on both local dev machine and HPC
WORKSHOP_DIR = Path(".").resolve().parent
DATA_DIR = WORKSHOP_DIR / "data"
if not DATA_DIR.exists():
    # Fallback: try the cv-workshop repo location
    _alt = Path.home() / "dejan_dev" / "ai-factory" / "cv-workshop" / "workshop" / "data"
    if _alt.exists():
        DATA_DIR = _alt

IMAGES_DIR = DATA_DIR / "synthetic_ppe"
DATASET_DIR = DATA_DIR / "ppe_dataset_v2"
ERROR_ANALYSIS_DIR = DATA_DIR / "error_analysis"

print(f"\nData dir: {DATA_DIR}")
print(f"Images:   {IMAGES_DIR} ({'found' if IMAGES_DIR.exists() else 'NOT FOUND'})")
print(f"Dataset:  {DATASET_DIR} ({'found' if DATASET_DIR.exists() else 'NOT FOUND'})")

In [ ]:
# Load SAM3 teacher model via HuggingFace Transformers
from transformers import Sam3Model, Sam3Processor

print("Loading SAM3 teacher model (facebook/sam3)...")
print("This may take a minute on first run (downloading ~2.5 GB).")

processor = Sam3Processor.from_pretrained("facebook/sam3")
model = Sam3Model.from_pretrained("facebook/sam3").to(device)

# Count parameters
param_count = sum(p.numel() for p in model.parameters())
print(f"\nSAM3 teacher loaded on {device}")
print(f"Parameters: {param_count / 1e9:.1f}B ({param_count:,})")

---
## Step 2: Auto-Label One Image

The teacher examines one image at a time. We prompt it with a text description
of what to find -- just like talking to a search engine.

In [ ]:
# Pick a sample image
img_path = IMAGES_DIR / "easy" / "easy_01.jpg"
if not img_path.exists():
    # Fallback: find any .jpg in the images directory
    candidates = list(IMAGES_DIR.rglob("*.jpg"))
    if candidates:
        img_path = candidates[0]
    else:
        raise FileNotFoundError(f"No images found in {IMAGES_DIR}")

img = Image.open(img_path).convert("RGB")
print(f"Image: {img_path.name}  ({img.width}x{img.height})")

# Run SAM3 with text prompt
PROMPT = "helmet"  # The minimal prompt -- we'll explain why later
inputs = processor(images=img, text=PROMPT, return_tensors="pt").to(device)

with torch.no_grad():
    outputs = model(**inputs)

results = processor.post_process_instance_segmentation(
    outputs,
    threshold=0.3,
    mask_threshold=0.0,
    target_sizes=[(img.height, img.width)],
)[0]

boxes = results.get("boxes", [])
scores = results.get("scores", [])
if hasattr(boxes, "tolist"):
    boxes = boxes.tolist()
if hasattr(scores, "tolist"):
    scores = scores.tolist()

print(f"\nPrompt: \"{PROMPT}\"")
print(f"Teacher found {len(boxes)} objects:")
for i, (box, score) in enumerate(zip(boxes, scores)):
    x1, y1, x2, y2 = box
    print(f"  [{i}] helmet ({score:.2f}): [{x1:.0f}, {y1:.0f}, {x2:.0f}, {y2:.0f}]")

# Visualize
fig, ax = plt.subplots(1, 1, figsize=(10, 8))
ax.imshow(np.array(img))
for box, score in zip(boxes, scores):
    x1, y1, x2, y2 = box
    rect = patches.Rectangle((x1, y1), x2 - x1, y2 - y1,
                              linewidth=2, edgecolor="lime", facecolor="none")
    ax.add_patch(rect)
    ax.text(x1, y1 - 5, f"helmet {score:.2f}", color="lime", fontsize=10,
            fontweight="bold", bbox=dict(boxstyle="round,pad=0.2",
            facecolor="black", alpha=0.7))
ax.set_title(f"SAM3 Teacher Detections: \"{PROMPT}\" on {img_path.name}", fontsize=12)
ax.axis("off")
plt.tight_layout()
plt.show()

---
## Step 3: Convert Detections to YOLO Format

YOLO label format: one `.txt` file per image, one line per object:

```
class_id  center_x  center_y  width  height
```

All coordinates are **normalized** (0.0 to 1.0) relative to image dimensions.
This makes labels resolution-independent.

In [ ]:
# Convert pixel coordinates to YOLO format
img_w, img_h = img.size
print(f"Image size: {img_w} x {img_h}")
print(f"\nPixel coords  \u2192  YOLO format (class_id cx cy w h):")
print("=" * 60)

yolo_lines = []
for box in boxes:
    x1, y1, x2, y2 = box
    cx = ((x1 + x2) / 2) / img_w
    cy = ((y1 + y2) / 2) / img_h
    w = (x2 - x1) / img_w
    h = (y2 - y1) / img_h
    line = f"0 {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}"
    yolo_lines.append(line)
    print(f"  [{x1:.0f},{y1:.0f},{x2:.0f},{y2:.0f}]  \u2192  {line}")

print(f"\nThis is exactly what a human annotator would produce.")
print(f"Same format. Same structure. But it took milliseconds, not minutes.")

---
## Step 4: Inspect Pre-Generated Labels

The full pipeline labels all 91 images with multiple prompts (`"helmet"`, `"person"`,
`"person not wearing a hard hat"`), resolves conflicts, deduplicates, and writes
one `.txt` file per image. Let's look at a real generated label file.

In [ ]:
# Display a pre-generated label file from the dataset
label_dir = DATASET_DIR / "labels" / "train"

if label_dir.exists():
    label_files = sorted(label_dir.glob("*.txt"))
    # Filter out any CLAUDE.md or non-label files
    label_files = [f for f in label_files if f.suffix == ".txt"]

    if label_files:
        sample_label = label_files[0]
        content = sample_label.read_text()
        lines = content.strip().split("\n")

        print(f"Label file: {sample_label.name}")
        print("=" * 60)
        print(content)
        print(f"\n{len(lines)} objects labeled in this image")
        print(f"\nClass mapping: 0=hardhat, 1=no_hardhat, 2=person")

        # Show a few more
        print(f"\n\n--- Dataset overview ---")
        print(f"Total label files (train): {len(label_files)}")
        val_labels = list((DATASET_DIR / "labels" / "val").glob("*.txt")) if (DATASET_DIR / "labels" / "val").exists() else []
        print(f"Total label files (val):   {len(val_labels)}")
    else:
        print(f"No .txt label files found in {label_dir}")
else:
    print(f"Dataset directory not found: {DATASET_DIR}")
    print("This is expected if running on HPC without the pre-generated dataset.")
    print("\nExample label file content:")
    print("0 0.416100 0.146752 0.065416 0.082411")
    print("1 0.559630 0.245942 0.147133 0.243909")
    print("2 0.559630 0.530502 0.147133 0.813028")
    print("2 0.409229 0.521510 0.151518 0.836509")

In [ ]:
# Show the dataset structure
print("YOLO Dataset Structure:")
print("=" * 40)
print(f"""ppe_dataset_v2/
\u251c\u2500\u2500 dataset.yaml          # Class names + paths
\u251c\u2500\u2500 images/
\u2502   \u251c\u2500\u2500 train/           # {len(label_files) if label_dir.exists() else 64} training images
\u2502   \u2514\u2500\u2500 val/             # {len(val_labels) if label_dir.exists() else 17} validation images
\u2514\u2500\u2500 labels/
    \u251c\u2500\u2500 train/           # One .txt per training image
    \u2514\u2500\u2500 val/             # One .txt per validation image""")

# Show dataset.yaml if available
yaml_path = DATASET_DIR / "dataset.yaml"
if yaml_path.exists():
    print(f"\n\ndataset.yaml contents:")
    print("-" * 40)
    print(yaml_path.read_text())
else:
    print(f"\n\ndataset.yaml (expected contents):")
    print("-" * 40)
    print("""path: /path/to/ppe_dataset_v2
train: images/train
val: images/val

names:
  0: hardhat
  1: no_hardhat
  2: person

nc: 3""")

---
## The Student Learns From These Labels

The student model (YOLO26n) has never seen the teacher. It only sees:
- The images
- The `.txt` label files

From the student's perspective, these labels are indistinguishable from human annotations.
Same format. Same coordinate system. The student doesn't know -- or care -- that a
2.9-billion-parameter foundation model wrote them.

```python
from ultralytics import YOLO

student = YOLO("yolo26n.pt")  # 2.4M parameters, pre-trained on COCO

# Stage 1: Freeze backbone, train head (10 epochs)
student.train(data="dataset.yaml", epochs=10, freeze=10, lr0=0.01)

# Stage 2: Unfreeze everything, fine-tune (40 epochs)
student.train(data="dataset.yaml", epochs=40, lr0=0.001)

# Stage 3: Extended training with low LR (50 more epochs)
student.train(data="dataset.yaml", epochs=50, lr0=0.0005)
```

We won't run this live (it takes 30+ minutes). Let's look at the pre-baked results.

---
## Pre-Baked Training Results

**Model:** YOLO26n (2.4M parameters)  
**Data:** 64 training images, 17 validation images  
**Source:** All synthetic (Gemini 3 Pro) + auto-labeled (YOLOE / Grounding DINO)

### Three-Stage Training Journey

| Stage | Strategy | mAP50 | Notes |
|:-----:|----------|:-----:|-------|
| 1 | Frozen backbone (10 epochs) | 0.436 | Head learns new classes |
| 2 | Full fine-tuning (40 epochs) | 0.505 | End-to-end optimization |
| 3 | Extended training (50 more epochs) | **0.633** | Low learning rate polish |

### Per-Class Breakdown

| Class | AP50 | Notes |
|-------|:----:|-------|
| person | 0.900 | Approaching production quality |
| hardhat | 0.490 | Improved 49% from v1 |
| no_hardhat | 0.510 | The hard class (detecting *absence*) |

Zero human annotations. Zero manual labeling. 64 synthetic images.
Person detection at 0.90 -- approaching production quality.

In [ ]:
# Display pre-generated confusion matrix and results plots
from IPython.display import display

confusion_path = ERROR_ANALYSIS_DIR / "confusion_matrix_val.png"

if confusion_path.exists():
    fig, ax = plt.subplots(1, 1, figsize=(8, 6))
    cm_img = Image.open(confusion_path)
    ax.imshow(np.array(cm_img))
    ax.set_title("Validation Confusion Matrix (V2 Pipeline)", fontsize=13)
    ax.axis("off")
    plt.tight_layout()
    plt.show()
    print("\nKey observation: Zero inter-class misclassifications.")
    print("The student never confuses helmet with no_hardhat.")
    print("All errors are missed detections (false negatives), not wrong labels.")
else:
    print(f"Confusion matrix not found at: {confusion_path}")
    print("\nPre-computed results summary:")
    print("  - Zero inter-class misclassifications")
    print("  - All errors are false negatives (missed detections)")
    print("  - person recall: 71.6%")
    print("  - hardhat recall: 58.3%")
    print("  - no_hardhat recall: 25.5% (the hard class)")

---
## Speed Comparison: Teacher vs Student

Let's measure the actual inference speed difference on the same image.

In [ ]:
# ---- Teacher timing (SAM3) ----
test_img = Image.open(img_path).convert("RGB")
N_RUNS = 5

# Warm-up
warmup_inputs = processor(images=test_img, text="helmet", return_tensors="pt").to(device)
with torch.no_grad():
    _ = model(**warmup_inputs)

# Time the teacher
teacher_times = []
for _ in range(N_RUNS):
    t_inputs = processor(images=test_img, text="helmet", return_tensors="pt").to(device)
    if device == "cuda":
        torch.cuda.synchronize()
    start = time.perf_counter()
    with torch.no_grad():
        _ = model(**t_inputs)
    if device == "cuda":
        torch.cuda.synchronize()
    teacher_times.append((time.perf_counter() - start) * 1000)

avg_teacher = sum(teacher_times) / len(teacher_times)
print(f"Teacher (SAM3):")
print(f"  Average: {avg_teacher:.1f} ms")
print(f"  FPS:     {1000 / avg_teacher:.0f}")
print(f"  Runs:    {[f'{t:.1f}' for t in teacher_times]}")

In [ ]:
# ---- Student timing (YOLO26n) ----
try:
    from ultralytics import YOLO

    student = YOLO("yolo26n.pt")

    # Warm-up
    _ = student(str(img_path), verbose=False)

    student_times = []
    for _ in range(N_RUNS):
        if device == "cuda":
            torch.cuda.synchronize()
        start = time.perf_counter()
        _ = student(str(img_path), verbose=False)
        if device == "cuda":
            torch.cuda.synchronize()
        student_times.append((time.perf_counter() - start) * 1000)

    avg_student = sum(student_times) / len(student_times)

    print(f"Student (YOLO26n):")
    print(f"  Average: {avg_student:.1f} ms")
    print(f"  FPS:     {1000 / avg_student:.0f}")
    print(f"  Runs:    {[f'{t:.1f}' for t in student_times]}")

    print(f"\n{'=' * 40}")
    print(f"Speedup: {avg_teacher / avg_student:.1f}x faster")
    print(f"Teacher: {avg_teacher:.1f} ms  |  Student: {avg_student:.1f} ms")
    print(f"{'=' * 40}")

    # Visual comparison bar chart
    fig, ax = plt.subplots(figsize=(8, 3))
    models = ["SAM3 (Teacher)", "YOLO26n (Student)"]
    times = [avg_teacher, avg_student]
    colors = ["#e74c3c", "#27ae60"]
    bars = ax.barh(models, times, color=colors, height=0.5)
    ax.set_xlabel("Inference Time (ms)", fontsize=11)
    ax.set_title(f"Speed Comparison ({device.upper()}) -- {avg_teacher / avg_student:.1f}x speedup", fontsize=12)
    for bar, t in zip(bars, times):
        ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height() / 2,
                f"{t:.1f} ms", va="center", fontsize=11, fontweight="bold")
    ax.set_xlim(0, max(times) * 1.3)
    plt.tight_layout()
    plt.show()

except Exception as e:
    print(f"Could not load YOLO26n student model: {e}")
    print(f"\nFallback benchmark numbers (from T4 GPU):")
    print(f"  Teacher (YOLOE-L): 6.2 ms (161 FPS)")
    print(f"  Student (YOLO26n): 1.7 ms (588 FPS)")
    print(f"  Speedup: 3.6x")

---
## The Prompt Engineering Story

This is the experiment that changed the entire pipeline.

We tested **3 different text prompts** on the same 75 images. Same model (Grounding DINO).
Same confidence threshold. Same everything. Just different words.

| Prompt | Helmets Found | Avg Confidence |
|--------|:-------------:|:--------------:|
| `"hard hat. safety helmet. person."` | 190 | 0.46 |
| `"yellow hard hat. orange helmet. white helmet. person."` | 193 | 0.37 |
| `"helmet. person."` | **424** | **0.61** |

The minimal prompt found **2.2x more helmets** at **32% higher confidence**.

Only **4% of images** gave consistent results across all three prompts.
For the other 96%, the prompt choice determined what the teacher saw and what it missed.

**Three words. That's the difference between missing half your helmets and finding them.**

### Why Does This Matter for Distillation?

Every missed helmet in the teacher's output = a training error for the student.

```
"hard hat. safety helmet. person." misses 234 helmets
  -> 234 workers WITH helmets labeled as "no_hardhat"
  -> Student learns: "heads with helmets = no helmet"
  -> Student is confused. Performance caps at mAP50 ~ 0.55

"helmet. person." finds those 234 helmets
  -> Correct labels
  -> Student learns correctly
  -> Performance jumps to mAP50 = 0.633
```

The prompt IS the annotation quality.  
In open-vocabulary distillation, **prompt engineering IS data engineering.**

---
## The Golden Rule

```
Foundation models are concept detectors, not logic engines.

  Detect THINGS with the model    -> "helmet", "person"
  Detect RELATIONSHIPS with code  -> if no overlap: "non-compliant"

Simple nouns + post-processing    -> 71% accuracy
Compliance prompts directly       -> 13% accuracy
```

If you try to teach the model compliance -- `"person not wearing a helmet"` -- you get
13% accuracy. The teacher can't see negation, so the student inherits nonsense labels.

Instead: detect the **things**. `"Helmet."` `"Person."` Two simple nouns. Then write ten
lines of code to check overlap. If a person bounding box has no overlapping helmet bounding
box -- non-compliant. 71% accuracy. Five times better.

Every failure we saw today -- the negation blindness, the bag-of-words collapse, the prompt
sensitivity -- they all point here. The model is extraordinary at recognizing concepts it has
seen in 400 million captions. It is hopeless at reasoning about relationships between those
concepts. That is YOUR job.

---
## Label Quality > Data Quantity

We proved this with a **data scaling experiment**. Same prompt, increasing number of
training images:

| Training Images | mAP50 | Trend |
|:---------------:|:-----:|-------|
| 15 | 0.44 | Rising |
| 30 | 0.56 | Rising |
| 45 | 0.56 | **Plateau** |
| 59 | 0.51 | **Regression** |

Fitted curve: saturating exponential, **R^2 = 0.97** (tight fit).  
Asymptote at **mAP50 ~ 0.53** -- this IS the ceiling with those labels.

Then we **fixed the prompt** (same 64 images):

| Change | mAP50 |
|--------|:-----:|
| Old prompt, 59 images | 0.51 |
| **New prompt, 64 images** | **0.633** |

More data hit a wall. Better labels broke through it.

The teacher's prompt is your data quality lever. Invest your time in prompt engineering
and label validation, not in generating thousands more images.

In [ ]:
# Scaling curve visualization
# Try to load pre-generated plot; otherwise render from data

scaling_plot_path = DATA_DIR / "scaling_prediction.png"

if scaling_plot_path.exists():
    fig, ax = plt.subplots(figsize=(10, 6))
    sc_img = Image.open(scaling_plot_path)
    ax.imshow(np.array(sc_img))
    ax.axis("off")
    ax.set_title("Data Scaling Curve: Label Quality > Data Quantity", fontsize=13)
    plt.tight_layout()
    plt.show()
else:
    # Render from experiment data
    data_points = [15, 30, 45, 59]
    map50_values = [0.44, 0.56, 0.56, 0.51]

    # Fitted saturating curve: f(x) = a * (1 - exp(-b * x))
    # a ~ 0.527, b ~ 0.065 (from experiment report, R^2 = 0.97)
    a, b = 0.527, 0.065
    x_fit = np.linspace(5, 80, 100)
    y_fit = a * (1 - np.exp(-b * x_fit))

    fig, ax = plt.subplots(figsize=(10, 6))

    # Plot fitted curve
    ax.plot(x_fit, y_fit, "--", color="#95a5a6", linewidth=2,
            label=f"Fitted curve (asymptote = {a:.3f}, R\u00b2 = 0.97)")
    ax.axhline(y=a, color="#e74c3c", linestyle=":", alpha=0.7,
               label=f"Ceiling with old labels ({a:.3f})")

    # Plot data points
    ax.scatter(data_points, map50_values, s=100, color="#3498db", zorder=5,
              label="Old prompt experiments")
    for x, y in zip(data_points, map50_values):
        ax.annotate(f"{y:.2f}", (x, y), textcoords="offset points",
                   xytext=(0, 12), ha="center", fontsize=10, fontweight="bold")

    # Plot the breakthrough point
    ax.scatter([64], [0.633], s=200, color="#27ae60", marker="*", zorder=5,
              label="Fixed prompt (0.633)")
    ax.annotate("Fixed prompt\n0.633", (64, 0.633), textcoords="offset points",
               xytext=(10, -20), ha="left", fontsize=11, fontweight="bold",
               color="#27ae60",
               arrowprops=dict(arrowstyle="->", color="#27ae60"))

    ax.set_xlabel("Number of Training Images", fontsize=12)
    ax.set_ylabel("mAP50", fontsize=12)
    ax.set_title("Data Scaling: More Data Hit a Wall. Better Labels Broke Through.",
                fontsize=13, fontweight="bold")
    ax.set_ylim(0.3, 0.75)
    ax.set_xlim(0, 80)
    ax.legend(loc="lower right", fontsize=10)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    print("\nThe ceiling with the old prompt was mAP50 = 0.527.")
    print("Fixing three words in the prompt broke through to 0.633.")
    print("No additional images were needed.")

---
## Tomorrow: Make It Yours

**Day 2 (16:00-18:00)**

You will:
1. Generate synthetic images for YOUR problem (Gemini API)
2. Run YOLOE on YOUR images with YOUR prompts
3. See where it fails -- error analysis is the real skill
4. Distill to YOLO26n -- YOUR model, YOUR classes
5. Compare model sizes: which fits YOUR deployment target?

**What to think about tonight:**
- What do you want to detect?
- Where will the model run? (cloud, edge, mobile)
- What is your FPS requirement?
- What is your accuracy threshold?
- Does licensing matter for your use case?

---
## Key Takeaways

1. **Distillation pipeline**: Large teacher auto-labels images, small student trains on those labels. Teacher retires, student deploys.

2. **Speed matters**: YOLO26n is 3.6x faster than YOLOE-L. On edge hardware, that is the difference between real-time and dropped frames.

3. **Prompt engineering is data engineering**: The minimal prompt `"helmet. person."` found 2.2x more helmets than the specific prompt. Three words changed the mAP50 from 0.55 to 0.633.

4. **The Golden Rule**: Foundation models are concept detectors, not logic engines. Detect things with the model. Check relationships with code.

5. **Label quality > data quantity**: A saturating curve (R^2 = 0.97) proved that more data alone could not break through mAP50 = 0.53. Better labels broke through to 0.633.

6. **Zero human annotations**: Every label in this pipeline was generated by a machine. Person detection reached 0.90 AP -- approaching production quality from 64 synthetic images.